In [ ]:
# Customer Segmentation

In [ ]:
import sys
import os
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))

from sklearn.preprocessing import PowerTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from src.data_processing import (
    load_data,
    clean_data,
    validate_data
)

from src.rfm_calculator import (
    calculate_rfm,
    calculate_rfm_scores,
    segment_customers
)

In [ ]:
# Load dataset
df_raw = load_data('../data/raw/online_retail.csv')

# Clean dataset
df = clean_data(df_raw)

# Validate cleaned data
validate_data(df)

# Generate RFM table
rfm = calculate_rfm(df)

# Generate RFM scores
rfm = calculate_rfm_scores(rfm)

# Assign business segments
rfm = segment_customers(rfm)

# Preview
rfm.head()

In [ ]:
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].copy()

rfm_log['Frequency'] = np.log1p(rfm_log['Frequency'])
rfm_log['Monetary']  = np.log1p(rfm_log['Monetary'])

In [ ]:
# Select clustering features
rfm_features = rfm[['Recency', 'Frequency', 'Monetary']].copy()

# Handle skewness + scaling
transformer = PowerTransformer(method='yeo-johnson')

scaled_features = transformer.fit_transform(rfm_features)

print("Feature transformation completed.")

In [ ]:
# Build clustering model
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

# Assign clusters
rfm['Cluster'] = kmeans.fit_predict(scaled_features)

# Evaluate clustering quality
score = silhouette_score(
    scaled_features,
    rfm['Cluster']
)

print(f"Silhouette Score: {score:.3f}")

# Preview clustered data
rfm.head()

In [ ]:
cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).round(2)

cluster_summary